# Lesson 03 - Agentic Design Patterns

U ovoj lekciji istražujemo tri temeljna obrasca dizajna za izgradnju učinkovitih AI agenata:

1. **Jasne upute za agente** — Izrada preciznih, ulogom definiranih upita koji usmjeravaju ponašanje agenta  
2. **Strukturirani izlaz s Pydantic modelima** — Osiguravanje da agenti vraćaju predvidive, validirane podatke  
3. **Agenti s jednom odgovornošću** — Dizajniranje fokusiranih agenata koji svaki rade jednu stvar dobro  

Svaki ćemo obrazac primijeniti na scenarij **preporučitelja putničkih odredišta**, postupno gradeći sustav koji može predlagati odredišta, provjeravati dostupnost i rukovati logistikom.


## Postavljanje


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Uzorak 1: Jasne upute za agenta

Najutjecajniji uzorak je također i najjednostavniji: pisanje jasnih, detaljnih uputa za vašeg agenta.

Dobre upute definiraju:
- **Tko** je agent (persona i ton)
- **Što** treba raditi (odgovornosti korak po korak)
- **Kako** se treba ponašati (ograničenja i stil)

Ispod stvaramo agenta putničkog conciergea s eksplicitnim uputama koje oblikuju svaki odgovor koji daje.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Uzorak 2: Strukturirani izlaz s Pydantic modelima

Tekst slobodnog oblika koristan je za razgovor, ali sustavi niže razine trebaju strukturirane podatke.
Uparivanjem **Pydantic modela** s **alatom funkcije**, možemo:

- Definirati točan shemu za izlaz agenta
- Automatski provjeravati valjanost odgovora
- Pouzdano integrirati rezultate agenta u logiku aplikacije

Također uvodimo alat koji vraća detalje o odredištu kako bi agent temeljio svoje preporuke na stvarnim podacima.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Uzorak 3: Agent s Jedinstvenom Odgovornošću

Složeni zadaci imaju koristi od podjele rada među više fokusiranih agenata, svaki s jednom odgovornošću:

- **Stručnjak za odredište** koji zna o mjestima i dostupnosti
- **Planer logistike** koji upravlja letovima, hotelima i itinererima

Ovo odražava načelo inženjerstva softvera *odvajanja briga* — svaki je agent lakše testirati, održavati i poboljšavati neovisno.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Sažetak

U ovoj lekciji primijenili smo tri agentna obrasca dizajna na scenarij preporučitelja putovanja:

| Obrasc | Ključna ideja | Prednost |
|---|---|---|
| **Jasne upute** | Definiranje persone, odgovornosti i ograničenja unaprijed | Dosljedno, u skladu s brendom ponašanje agenta |
| **Strukturirani izlaz** | Koristiti Pydantic modele kao format odgovora | Validirani, strojno čitljivi rezultati |
| **Jedna odgovornost** | Dati svakom agentu jedan fokusirani zadatak | Jednostavnije testiranje, održavanje i sastavljanje |

Ovi obrasci se prirodno kombiniraju — možete spojiti jasne upute sa strukturiranim izlazom unutar agenta s jednom odgovornošću za izgradnju robusnih, proizvodnih sustava.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Izjava o odricanju odgovornosti**:
Ovaj dokument je preveden korištenjem AI prevoditeljske usluge [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili nepravilnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za kritične informacije preporuča se profesionalni prijevod od strane ljudskog stručnjaka. Ne odgovaramo za eventualne nesporazume ili kriva tumačenja proizašla iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
